In [ ]:
from datasets import load_dataset

ds = load_dataset("code-search-net/code_search_net", "all")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00002.parquet:   0%|          | 0.00/1.01G [00:00<?, ?B/s]

all/train-00001-of-00002.parquet:   0%|          | 0.00/774M [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/96.1M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/85.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1880853 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100529 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/89154 [00:00<?, ? examples/s]

In [ ]:
# 1. Filter the dataset to only keep Python code
print("Filtering for Python examples...")
python_ds = ds['train'].filter(lambda example: example['language'] == 'python')

print(f"Total Python examples ready for training: {len(python_ds)}")

# 2. Let's look at the exact data structure of the very first Python example
print("\n--- The English Explanation (What the model reads) ---")
print(python_ds[0]['func_documentation_string'])

print("\n--- The Python Code (What the model learns to write) ---")
print(python_ds[0]['func_code_string'])

Filtering for Python examples...


Filter:   0%|          | 0/1880853 [00:00<?, ? examples/s]

Total Python examples ready for training: 412178

--- The English Explanation (What the model reads) ---
Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution

--- The Python Code (What the model learns to write) ---
def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        """
        import scipy

        start = self._start_time
        seg = 1 - self.segmentation.astype(np.int8)
        self.stats["low level object voxels"] = np.sum(seg)
        self.stats["low level image voxels"] = np.prod(seg.shape)
        # in seg is now stored low resolution segmentation
        # back to normal parameters
        # step 2: discontinuity localization
        # self.segparams = sparams_hi
        seg_border = scipy.ndimage.filters.laplace(seg, mode="constant")
        logger.debug("seg_border

In [ ]:
from transformers import AutoTokenizer

# 1. Load a standard Tokenizer
# We use 'gpt2' here because it is fast, standard, and has a vocab size of 50,257
# which matches our Baby Llama configuration perfectly.
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token # Fix to prevent padding errors

# 2. Define the Prompt Format
def format_and_tokenize(examples):
    formatted_texts = []
    # We zip them together to format a batch at a time
    for doc, code in zip(examples['func_documentation_string'], examples['func_code_string']):
        # We format it exactly like a real Python file:
        # """
        # English description here
        # """
        # def python_code_here(): ...
        prompt = f'"""\n{doc}\n"""\n{code}'
        formatted_texts.append(prompt)

    # Convert the text into numbers (tokens).
    # max_length=256 keeps the memory usage low so Colab doesn't crash.
    return tokenizer(formatted_texts, truncation=True, padding="max_length", max_length=256)

# 3. Slice and Tokenize
print("Taking a slice of 10,000 examples for our Colab test run...")
small_python_ds = python_ds.select(range(10000))

print("Converting text to tokens (this takes about 10-20 seconds)...")
# The .map() function applies our tokenizer to the whole dataset super fast
tokenized_ds = small_python_ds.map(
    format_and_tokenize,
    batched=True,
    remove_columns=small_python_ds.column_names # Remove the old text columns to save RAM
)

print(f"\nSuccess! You now have {len(tokenized_ds)} fully processed examples ready for the neural network.")

# Let's peek at the actual numbers the model will see for the first example
print("First 20 tokens of example 1:", tokenized_ds[0]['input_ids'][:20])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Taking a slice of 10,000 examples for our Colab test run...
Converting text to tokens (this takes about 10-20 seconds)...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]


Success! You now have 10000 fully processed examples ready for the neural network.
First 20 tokens of example 1: [37811, 198, 22362, 1920, 19936, 14834, 287, 4308, 286, 1877, 6323, 2939, 10618, 341, 13, 198, 220, 220, 220, 220]


In [ ]:
import torch
from transformers import LlamaConfig, LlamaForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# 1. The Baby Llama Architecture
# We are keeping the layers and dimensions small so it trains quickly on Colab.
config = LlamaConfig(
    vocab_size=50257,      # Matches the GPT-2 tokenizer we used
    hidden_size=512,       # Small embedding size
    intermediate_size=1024,
    num_hidden_layers=4,   # Just 4 layers for this test run
    num_attention_heads=8,
    max_position_embeddings=256, # Matches our tokenization max_length
)

print("Initializing the untaught Baby Llama...")
model = LlamaForCausalLM(config)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on: {device}")
print(f"Total Parameters: {sum(p.numel() for p in model.parameters()) / 1_000_000:.1f} Million")

# 2. Data Collator
# This automatically formats our tokenized text into training batches and creates the "labels"
# (shifting the text by one token so the model learns to predict the next word).
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


# 3. Training Arguments (Optimized for Free Colab)
training_args = TrainingArguments(
    output_dir="./baby-llama-python",
    num_train_epochs=1,              # 1 pass through the 10000 examples
    per_device_train_batch_size=8,   # Small batch size to prevent RAM crashes
    save_steps=500,                  # Save checkpoints along the way
    logging_steps=100,               # Print progress every 100 steps
    learning_rate=5e-4,
    fp16=torch.cuda.is_available(),  # Use mixed precision for a massive speedup on the T4 GPU
    report_to="none"                 # Disables external logging trackers
)

# 4. The Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
)

# 5. START TRAINING!
print("\nStarting the training loop! Watch the Loss go down...")
trainer.train()

# 6. Save the final model
trainer.save_model("./baby-llama-python-final")
print("Training complete and model saved!")

Initializing the untaught Baby Llama...
Model loaded on: cuda
Total Parameters: 62.0 Million

Starting the training loop! Watch the Loss go down...


Step,Training Loss
100,4.861549
200,3.903436
300,3.547794
400,3.285104
500,3.226941
600,3.099576
700,3.034812
800,2.917303
900,2.899549
1000,2.826266


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and model saved!


In [ ]:
import torch
from transformers import AutoTokenizer, LlamaForCausalLM

# --- THE FIX: Load the exact tokenizer we used during training ---
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# 1. Load your newly trained brain
model_path = "./baby-llama-python-final"
print("Waking up the Baby Llama...")
trained_model = LlamaForCausalLM.from_pretrained(model_path)
trained_model.to("cuda" if torch.cuda.is_available() else "cpu")

# 2. Set up the prompt exactly how we formatted the training data
prompt = '"""\nWrite a function to add two numbers together.\n"""\ndef'

# 3. Tokenize the prompt
inputs = tokenizer(prompt, return_tensors="pt").to(trained_model.device)

# 4. Generate the code!
print("Generating code...\n")
outputs = trained_model.generate(
    **inputs,
    max_new_tokens=50,       # Let it write up to 50 tokens
    temperature=0.7,         # A little creativity
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# 5. Decode the numbers back into text
generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_code)

Loading Tokenizer...
Waking up the Baby Llama...


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

Generating code...

"""
Write a function to add two numbers together.
"""
def load_tuple(self, service_id, obj, method=None):
        """
        Return a class to the given name.
       


In [ ]:
!pip install -q accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 47.4 MB/s eta 0:00:00


In [ ]:
!pip install -U bitsandbytes accelerate transformers peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 98.9 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install -U torch torchvision torchaudio
!pip install -q accelerate peft bitsandbytes transformers trl datasets

ERROR: Operation cancelled by user
^C


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

print("1. Downloading TinyLlama 1.1B...")
model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Safely compress the 1.1B model down to 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading model in 4-bit precision...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("\n2. Configuring LoRA Adapter...")
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)

print("\nSuccess! Adapter attached. Let's look at the math:")
peft_model.print_trainable_parameters()

1. Downloading TinyLlama 1.1B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading model in 4-bit precision...


model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]


2. Configuring LoRA Adapter...

Success! Adapter attached. Let's look at the math:
trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# ── CELL 1: GPU Memory Reset ──────────────────────────────────────────
import gc, torch

# Delete any leftover model objects from the crashed run
for var in ['model', 'peft_model', 'trainer']:
    if var in dir():
        del var

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print(f"Total VRAM: {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

Free VRAM: 11.38 GB
Total VRAM: 15.64 GB


In [ ]:
# ── CELL 2: Full Training ──────────────────────────────────────────────
import trl; print(trl.__version__)
import torch, os, gc
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# 1. Dataset
print("Downloading CodeAlpaca-20k...")
dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
small_dataset = dataset.select(range(2000))

def create_text_column(example):
    return {"text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"}

formatted_dataset = small_dataset.map(create_text_column)
print(f"Free VRAM before model load: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

# 2. Load in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,          # ← streams weights, avoids CPU OOM too
)
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token
print(f"Free VRAM after model load: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

# 3. LoRA
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,                             # ← Reduced from 16 to save memory
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# 4. Training args
training_args = SFTConfig(              # ← SFTConfig replaces TrainingArguments
    output_dir="./tinyllama-code-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    dataloader_pin_memory=False,
    max_length=256,
    dataset_text_field="text",
      # ← lives here in newer TRL
)
# 5. Trainer
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=formatted_dataset,
    args=training_args,
    # ← no max_seq_length here anymore
)

print("\nStarting training...")
trainer.train()
trainer.model.save_pretrained("./tinyllama-code-adapter")
print("\nDone! Adapter saved.")



0.29.0
Free VRAM before model load: 9.07 GB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Free VRAM after model load: 9.04 GB
trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.



Starting training...


Step,Training Loss
10,1.313202
20,0.972070
30,0.959998
40,0.831733
50,0.829584
60,0.811429
70,0.844248
80,0.779386
90,0.831155
100,0.825834



Done! Adapter saved.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("1. Waking up the Base Model...")
model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# We load the base model in 4-bit again to keep Colab happy
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("2. Attaching your custom LoRA brain...")
# This merges your newly trained weights with the original model!
model = PeftModel.from_pretrained(base_model, "./tinyllama-code-adapter")

# 3. Formulate the Prompt
# We MUST use the exact same template we taught it during training
instruction = "Write a Python script to print hello world."
prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

print("3. Generating code...\n")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 4. Generate the output
outputs = model.generate(
    **inputs,
    max_new_tokens=250,      # Give it enough room to write the logic
    temperature=0.7,         # A good balance of precision and creativity
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# 5. Decode and print!
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n=================== AI OUTPUT ===================")
print(generated_text)
print("=================================================")

1. Waking up the Base Model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=250) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2. Attaching your custom LoRA brain...
3. Generating code...


=================== AI OUTPUT ===================
### Instruction:
Write a Python script to print hello world.

### Response:
print("hello world")

### ### Instruction:
Write a Python script to print hello world.

### Response:
a = 11
b = 12
print(a * b)


In [ ]:
!pip install -q transformers peft torch accelerate huggingface_hub


In [ ]:
from huggingface_hub import login
login(token="----------")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_REPO = "mohit7739/tinyllama-python-coder-lora"  # your existing adapter
OUTPUT_REPO  = "mohit7739/tinyllama-python-coder"       # new merged model

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO)

print("Merging LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model = model.merge_and_unload()

print("Pushing merged model to HuggingFace...")
model.push_to_hub(OUTPUT_REPO)
tokenizer.push_to_hub(OUTPUT_REPO)
print("✅ Done! https://huggingface.co/" + OUTPUT_REPO)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...borjndn/model.safetensors:   2%|1         | 40.0MB / 2.20GB            

README.md: 0.00B [00:00, ?B/s]

✅ Done! https://huggingface.co/mohit7739/tinyllama-python-coder
